In [94]:
# 🚦 UrbanSense Motor Vehicle Analytics (MVA) Pipeline
### End-to-end processing pipeline for detecting Indian traffic infractions using YOLO11, ByteTrack, and EasyOCR.

In [95]:
# 1. Environment Setup & Hardware Check
!nvidia-smi
!pip install -q ultralytics easyocr roboflow kaggle lapx supervision

Sat Sep  5 20:35:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             31W /   70W |    2693MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [96]:
## Step 1: Model Training (Optimised)
# Training the YOLO11n model using a unified 5-class Indian traffic dataset. Hyperparameters are tuned for small object recovery (number plates) and bare-head detection.

In [ ]:
from ultralytics import YOLO

# Initialise fresh weights to prevent inheriting biases
model = YOLO("yolo11n.pt")

print("Commencing optimised training run...")
results = model.train(
    data="/content/mva_unified_dataset/dataset.yaml", # Ensure this path points to your unified dataset
    project="UrbanSense_MVA",
    name="mva_detector_optimised",

    # Scaling & Duration
    epochs=200,
    patience=30,
    imgsz=800,
    batch=8,

    # Optimisation
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,

    # Augmentations
    mosaic=1.0,
    close_mosaic=15,
    degrees=10.0,
    shear=2.0,
    scale=0.5,

    device=0,
    save=True
)

In [97]:
## Step 2: Configure Tracking
#Generates a forgiving ByteTrack configuration to maintain entity IDs on fast-moving vehicles.

In [98]:
tracker_yaml = """
tracker_type: bytetrack
track_high_thresh: 0.3
track_low_thresh: 0.1
match_thresh: 0.8
"""
with open("/content/custom_tracker.yaml", "w") as f:
    f.write(tracker_yaml)
print("Custom tracker configuration generated at '/content/custom_tracker.yaml'")

Custom tracker configuration generated at '/content/custom_tracker.yaml'


In [99]:
## Step 3: MVA Inference & Database Logging
#The core execution loop. Reads a traffic stream, tracks entities, evaluates spatial infractions (helmet, zebra crossing, wrong-way), executes OCR, and commits evidence to SQLite.

In [113]:
from google.colab import files
import os

print("Click 'Choose Files' below to select the video from your desktop:")
uploaded = files.upload()

for filename in uploaded.keys():
    # Automatically rename it so the pipeline script finds it
    os.rename(filename, "/content/traffic_sample.mp4")
    print(f"\n✅ Success! '{filename}' uploaded and renamed to 'traffic_sample.mp4'.")

Click 'Choose Files' below to select the video from your desktop:


Saving 35 Traffic Violations in 90 seconds  Red Signal  India  Dashcam Footage - Vimal Krishna's Logic (720p, h264).mp4 to 35 Traffic Violations in 90 seconds  Red Signal  India  Dashcam Footage - Vimal Krishna's Logic (720p, h264).mp4

✅ Success! '35 Traffic Violations in 90 seconds  Red Signal  India  Dashcam Footage - Vimal Krishna's Logic (720p, h264).mp4' uploaded and renamed to 'traffic_sample.mp4'.


In [117]:
tracker_yaml = """
tracker_type: bytetrack
track_high_thresh: 0.3
track_low_thresh: 0.1
match_thresh: 0.8
track_buffer: 30
fuse_score: True
"""
with open("/content/custom_tracker.yaml", "w") as f:
    f.write(tracker_yaml)
print("Updated custom_tracker.yaml with complete ByteTrack parameters.")

Updated custom_tracker.yaml with complete ByteTrack parameters.


In [119]:
import cv2
import re
import numpy as np
import easyocr
import sqlite3
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

# 1. Initialise Model & OCR
weights_path = "/content/runs/detect/UrbanSense_MVA/mva_detector_optimised/weights/best.pt"
model = YOLO(weights_path)
ocr_reader = easyocr.Reader(['en'], gpu=True)

# 2. Database Setup
evidence_dir = Path("/content/urbansense_evidence")
evidence_dir.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect('/content/urbansense_violations.db')
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS violations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        track_id INTEGER,
        violation_type TEXT,
        frame_idx INTEGER,
        licence_plate TEXT,
        owner_name TEXT,
        vehicle_model TEXT,
        rider_image_path TEXT,
        plate_image_path TEXT,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
    )
''')
conn.commit()

MOCK_RTO_DATABASE = {
    "MH02BT6482": {"owner": "Rahul Sharma", "model": "Honda Activa 6G"},
    "MH02DS9365": {"owner": "Priya Verma", "model": "Royal Enfield Classic 350"},
    "MH04J8401":  {"owner": "Amit Patel", "model": "Bajaj Pulsar 150"},
    "KA01AB1234": {"owner": "Ananya Roy", "model": "TVS Jupiter"}
}

def clean_plate_text(text):
    return re.sub(r'[^A-Z0-9]', '', text.upper())

def compute_iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea + 1e-6)

# 3. Video Processing Loop
input_video = "/content/traffic_sample.mp4"
cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    print(f"Error: Could not open {input_video}. Ensure the file exists in /content/")
else:
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25

    output_video_path = "/content/mva_processed_output.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    # Analytical Zones & Variables
    zebra_polygon = np.array([
        [int(width * 0.15), int(height * 0.70)], [int(width * 0.85), int(height * 0.70)],
        [int(width * 0.95), int(height * 0.85)], [int(width * 0.05), int(height * 0.85)]
    ], np.int32)

    trajectory_history = defaultdict(list)
    logged_violations = set()
    LEGAL_DIRECTION_Y = 1
    frame_idx = 0

    print("Commencing UrbanSense inference pipeline...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_idx += 1

        # Render Analytics Overlay
        overlay = frame.copy()
        cv2.polylines(overlay, [zebra_polygon], isClosed=True, color=(0, 165, 255), thickness=2)
        cv2.fillPoly(overlay, [zebra_polygon], color=(0, 165, 255))
        cv2.addWeighted(overlay, 0.25, frame, 0.75, 0, frame)

        # Using built-in bytetrack.yaml to prevent attribute configuration errors
        results = model.track(frame, tracker="bytetrack.yaml", persist=True, conf=0.35, imgsz=800, verbose=False)[0]
        vehicles, riders, helmets, no_helmets, plates = [], [], [], [], []

        if results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            track_ids = results.boxes.id.int().cpu().numpy()
            clss = results.boxes.cls.int().cpu().numpy()

            for box, t_id, cls in zip(boxes, track_ids, clss):
                cx, cy = int((box[0] + box[2]) / 2), int((box[1] + box[3]) / 2)
                entity = {"id": t_id, "box": box, "center": (cx, cy)}

                if cls == 0: vehicles.append(entity)
                elif cls == 1: riders.append(entity)
                elif cls == 2: helmets.append(entity)
                elif cls == 3: no_helmets.append(entity)
                elif cls == 4: plates.append(entity)

                # Basic Bounding Box Rendering
                cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (255, 255, 0), 2)

            # --- 3A. Inverse Helmet Violation Check ---
            for rider in riders:
                rx1, ry1, rx2, ry2 = rider["box"]
                head_box = [rx1, ry1, rx2, ry1 + 0.35 * (ry2 - ry1)] # Upper 35% of bounding box

                has_explicit_no_helmet = any(compute_iou(head_box, nh["box"]) > 0.05 for nh in no_helmets)
                has_detected_helmet = any(compute_iou(head_box, h["box"]) > 0.05 for h in helmets)

                if has_explicit_no_helmet or not has_detected_helmet:
                    event_key = (rider["id"], "NO_HELMET")
                    if event_key not in logged_violations:
                        logged_violations.add(event_key)

                        # Crop Evidence
                        rider_crop = frame[max(0, int(ry1)):min(height, int(ry2)), max(0, int(rx1)):min(width, int(rx2))]
                        rider_img_path = str(evidence_dir / f"rider_{rider['id']}_frame{frame_idx}.jpg")
                        if rider_crop.size > 0: cv2.imwrite(rider_img_path, rider_crop)

                        # Plate Mapping & OCR
                        associated_plate, plate_img_path = "UNKNOWN", "N/A"
                        for p in plates:
                            if compute_iou(rider["box"], p["box"]) > 0 or abs(ry2 - p["box"][1]) < 90:
                                px1, py1, px2, py2 = map(int, p["box"])
                                plate_crop = frame[max(0, py1):min(height, py2), max(0, px1):min(width, px2)]
                                if plate_crop.size > 0:
                                    plate_img_path = str(evidence_dir / f"plate_{rider['id']}_frame{frame_idx}.jpg")
                                    cv2.imwrite(plate_img_path, plate_crop)
                                    ocr_res = ocr_reader.readtext(plate_crop)
                                    if ocr_res:
                                        associated_plate = clean_plate_text(ocr_res[0][1])
                                        break

                        rto_record = MOCK_RTO_DATABASE.get(associated_plate, {"owner": "Unknown", "model": "N/A"})
                        cursor.execute('''
                            INSERT INTO violations (track_id, violation_type, frame_idx, licence_plate, owner_name, vehicle_model, rider_image_path, plate_image_path)
                            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                        ''', (int(rider["id"]), "NO_HELMET", frame_idx, associated_plate, rto_record["owner"], rto_record["model"], rider_img_path, plate_img_path))
                        conn.commit()

            # --- 3B. Trajectory Checks ---
            for entity in riders + vehicles:
                t_id, (cx, cy) = entity["id"], entity["center"]
                trajectory_history[t_id].append((cx, cy))

                if cv2.pointPolygonTest(zebra_polygon, (float(cx), float(cy)), False) >= 0:
                    if (t_id, "ZEBRA_CROSSING") not in logged_violations:
                        logged_violations.add((t_id, "ZEBRA_CROSSING"))
                        cursor.execute('INSERT INTO violations (track_id, violation_type, frame_idx) VALUES (?, ?, ?)', (int(t_id), "ZEBRA_CROSSING", frame_idx))
                        conn.commit()

                if len(trajectory_history[t_id]) >= 15:
                    dy = trajectory_history[t_id][-1][1] - trajectory_history[t_id][-15][1]
                    if dy * LEGAL_DIRECTION_Y < -25:
                        if (t_id, "WRONG_WAY") not in logged_violations:
                            logged_violations.add((t_id, "WRONG_WAY"))
                            cursor.execute('INSERT INTO violations (track_id, violation_type, frame_idx) VALUES (?, ?, ?)', (int(t_id), "WRONG_WAY", frame_idx))
                            conn.commit()

        out.write(frame)

    cap.release()
    out.release()
    conn.close()
    print(f"Video processing finished. Annotated file saved to: {output_video_path}")

Commencing UrbanSense inference pipeline...
Video processing finished. Annotated file saved to: /content/mva_processed_output.mp4


In [102]:
## Step 4: Frontend Developer Hand-off
#Generates a pre-populated synthetic database and placeholder evidence crops, bypassing the need for live video processing, then packages the assets into a downloadable zip archive for the dashboard developer.

In [120]:
import sqlite3
import pandas as pd
import os
import shutil
from pathlib import Path
from google.colab import files
from IPython.display import display

print("1. Verifying Database and Logged Violations...")

db_path = '/content/urbansense_violations.db'
if not os.path.exists(db_path):
    print(f"Error: Database file not found at {db_path}. Please ensure Step 3 (Inference Pipeline) has finished running first.")
else:
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query("SELECT id, track_id, violation_type, frame_idx, licence_plate, owner_name FROM violations ORDER BY id DESC", conn)
    conn.close()

    if df.empty:
        print("⚠️ The database is connected, but zero violations were logged during this video run.")
        print("Tip: Ensure your video features clear motorbike riders or boundary line crossings matching the analytical polygons.")
    else:
        print(f"✅ Successfully retrieved {len(df)} actual violation records from your video:\n")
        display(df.head(10))

    print("\n2. Packaging Database and Evidence Package for Handoff...")
    handoff_dir = "/content/handoff_package"
    if os.path.exists(handoff_dir): shutil.rmtree(handoff_dir)
    os.makedirs(handoff_dir, exist_ok=True)

    # Copy actual database
    shutil.copy(db_path, handoff_dir)

    # Copy actual evidence images folder if it exists
    evidence_dir = "/content/urbansense_evidence"
    if os.path.exists(evidence_dir):
        shutil.copytree(evidence_dir, f'{handoff_dir}/urbansense_evidence', dirs_exist_ok=True)

    zip_path = "/content/dashboard_backend_handoff"
    shutil.make_archive(zip_path, 'zip', handoff_dir)

    print("\n📦 Triggering download for 'dashboard_backend_handoff.zip'...")
    files.download(f'{zip_path}.zip')

1. Verifying Database and Logged Violations...
✅ Successfully retrieved 79 actual violation records from your video:



,id,track_id,violation_type,frame_idx,licence_plate,owner_name
0,79,356,NO_HELMET,2855,UNKNOWN,Unknown
1,78,354,WRONG_WAY,2838,None,None
2,77,354,NO_HELMET,2820,UNKNOWN,Unknown
3,76,337,NO_HELMET,2724,UNKNOWN,Unknown
4,75,334,NO_HELMET,2719,UNKNOWN,Unknown
5,74,307,NO_HELMET,2630,UNKNOWN,Unknown
6,73,309,NO_HELMET,2608,UNKNOWN,Unknown
7,72,303,NO_HELMET,2588,UNKNOWN,Unknown
8,71,294,NO_HELMET,2562,UNKNOWN,Unknown
9,70,291,NO_HELMET,2550,UNKNOWN,Unknown



2. Packaging Database and Evidence Package for Handoff...

📦 Triggering download for 'dashboard_backend_handoff.zip'...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>